# device-consistent-construct — ex2: infer device + dtype from an existing param via next(self.parameters())

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `device-consistent-construct`. Running the final beacon cell reports progress against the `PyTorch: Device-consistent tensor construction` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: Device-consistent tensor construction` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`device-consistent-construct`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "device-consistent-construct"
DD_SUBTOPIC = "PyTorch: Device-consistent tensor construction"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Inferring device + dtype from an existing param — `next(parameters())`

Ex1 read `device` + `dtype` directly off the input tensor `x`. The deepening move handles the case where you need to allocate scratch BEFORE you have an input — e.g. inside `__init__` after the module has been `.to(device)`-moved. The canonical idiom:

```python
def _ref_param(self):
    return next(self.parameters())  # first registered Parameter

def make_scratch(self, shape):
    ref = self._ref_param()
    return t.zeros(shape, device=ref.device, dtype=ref.dtype)
```

**Why `next(self.parameters())` is the standard trick.** All registered Parameters live on the same device after `.to(device)` — picking the first one is enough. This is what `torch.nn.Transformer` and HF's modeling code use internally when they need to materialize a mask or positional buffer without an input tensor in hand.

**Failure mode if the module has no parameters.** `next()` raises `StopIteration`. Guard with a `try/except` or `try` `next(self.buffers())` as fallback. The drill explicitly tests both the happy path AND the no-params guard.

### Exercise 2 — infer device + dtype from an existing param via next(self.parameters())

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the `next(self.parameters())` idiom to infer the module's current device + dtype without an input tensor in hand, and gracefully fall back when the module has no registered parameters.
> Keywords: device, dtype, parameters, next, scratch
> ```

**KCs targeted:** `next-self-parameters-ref`, `fallback-when-no-parameters`

Implement `ex2_make_scratch_module()`. Returns an `nn.Module` subclass instance with these methods:

1. `__init__(self)`:
   - `super().__init__()`.
   - Register `self.linear = nn.Linear(8, 4)` so the module has at least one Parameter.
2. `make_scratch(self, shape)`:
   - Get a reference param via `next(self.parameters())`. If `StopIteration` (no params), fall back to `torch.empty(0).device` (CPU) and `torch.empty(0).dtype` (float32).
   - Allocate and return `t.zeros(shape, device=ref.device, dtype=ref.dtype)`.

Constraints:
- DO NOT hardcode `device='cpu'` or `dtype=t.float32`. Always go through `next(self.parameters())` (or the fallback path) so the scratch tracks the module's CURRENT location after `.to(...)`.
- For the no-params fallback, the caller's tests will TEMPORARILY blank out the module's params — your method must handle that.
- Return value: a tensor with `shape == shape` (tuple-equal), `device == ref.device`, `dtype == ref.dtype`.

In [ ]:
def ex2_make_scratch_module():
    """Return a Module whose make_scratch(shape) infers device+dtype from its first param."""
    raise NotImplementedError()


def _test_ex2():
    import torch.nn as nn

    # === Default fresh module — params are on CPU float32 ===
    mod = ex2_make_scratch_module()
    assert isinstance(mod, nn.Module)
    assert hasattr(mod, 'make_scratch'), 'make_scratch method missing'
    assert any(True for _ in mod.parameters()), 'module must have at least one parameter'

    buf = mod.make_scratch((3, 5))
    assert isinstance(buf, t.Tensor)
    assert tuple(buf.shape) == (3, 5), f'shape wrong: {buf.shape}'
    assert buf.device.type == 'cpu', f'fresh module on CPU; buf must be CPU; got {buf.device}'
    assert buf.dtype == t.float32, f'default param dtype is float32; got {buf.dtype}'
    assert (buf == 0).all(), 'scratch must be zero-filled'

    # === After .to(dtype=float64), scratch follows the param's dtype ===
    mod = ex2_make_scratch_module()
    mod = mod.to(dtype=t.float64)
    buf = mod.make_scratch((2, 2))
    assert buf.dtype == t.float64, f'after to(float64), scratch must be float64; got {buf.dtype}'
    assert buf.device.type == 'cpu'

    # === After .to(dtype=float16), scratch follows again ===
    mod = ex2_make_scratch_module()
    mod = mod.to(dtype=t.float16)
    buf = mod.make_scratch((4,))
    assert buf.dtype == t.float16, f'after to(float16), scratch must be float16; got {buf.dtype}'

    # === Shape from a tuple, list, and torch.Size all work ===
    mod = ex2_make_scratch_module()
    for shape in [(2, 3), [4, 5], t.Size([6, 7])]:
        buf = mod.make_scratch(shape)
        assert tuple(buf.shape) == tuple(shape), f'shape arg {shape} mishandled: {buf.shape}'

    # === Empty shape → 0-D tensor ===
    mod = ex2_make_scratch_module()
    buf = mod.make_scratch(())
    assert buf.dim() == 0
    assert buf.item() == 0.0

    # === No-params fallback — strip the module's parameters and confirm fallback fires ===
    mod = ex2_make_scratch_module()
    # Remove the linear so the module has zero registered params.
    del mod.linear  # nn.Module __delattr__ removes from _modules and _parameters
    assert not any(True for _ in mod.parameters()), 'module should have no params after deletion'
    buf = mod.make_scratch((3,))
    assert isinstance(buf, t.Tensor)
    assert tuple(buf.shape) == (3,)
    assert buf.device.type == 'cpu', f'fallback must be CPU; got {buf.device}'
    assert buf.dtype == t.float32, f'fallback must be float32; got {buf.dtype}'

    # === After multiple .to() ops the scratch stays consistent with current state ===
    mod = ex2_make_scratch_module().to(dtype=t.float64).to(dtype=t.float32)
    buf = mod.make_scratch((2,))
    assert buf.dtype == t.float32, f'after double .to() the final dtype should win; got {buf.dtype}'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
import torch.nn as nn

class _ScratchModule(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(8, 4)

    def make_scratch(self, shape):
        try:
            ref = next(self.parameters())
            device, dtype = ref.device, ref.dtype
        except StopIteration:
            fallback = t.empty(0)
            device, dtype = fallback.device, fallback.dtype
        return t.zeros(shape, device=device, dtype=dtype)

def ex2_make_scratch_module():
    return _ScratchModule()
```

**`next(self.parameters())` is the canonical 'where is this module?' query.** All registered Parameters share the same device after `.to(device)` (PyTorch's `Module.to` walks all params). Picking the first one is sufficient — the rest agree.

**`StopIteration` fallback is real.** A module can have zero registered parameters (e.g. a pure-activation block, a custom Module wrapping a functional). Without the `try/except`, `next(self.parameters())` raises and your `make_scratch` is useless on those modules. Hugging Face's `PreTrainedModel._device` does the same fallback.

**Don't ALSO check `self.buffers()`.** Buffers (registered via `register_buffer`) live on the same device as parameters after `.to(...)`, but a buffer with `dtype=t.long` (e.g. position ids) would give your scratch the wrong dtype. Parameters-first, with an empty-tensor fallback, is the right precedence.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()